## 3. demand forecasting

train a lightgbm model to predict avg_available: the mean bikes at a station in a given hour.

the goal is 1h-ahead prediction. given the current station state, weather, and time of day, how many bikes will be at each station in the next hour. this is the most useful horizon for operational rebalancing decisions.

known limitation: training data had rebalancing-distorted hours removed. so the model predicts organic demand. during active rebalancing, predictions will diverge from observations. this is expected. a real production system would flag these divergences as potential rebalancing triggers rather than treating them as prediction errors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import pathlib
import warnings
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

FEATURES_PATH = pathlib.Path('../data/features.parquet')

if not FEATURES_PATH.exists():
    raise FileNotFoundError(f'{FEATURES_PATH} not found — run 02_features.ipynb first')

## 1. load features

In [ ]:
df = pd.read_parquet(FEATURES_PATH)
df['hour'] = pd.to_datetime(df['hour'])

print(f'shape:      {df.shape}')
print(f'stations:   {df["station_uid"].nunique():,}')
print(f'date range: {df["hour"].min()} to {df["hour"].max()}')

# drop rows missing lag features (first 168h per station at the start of the data)
before = len(df)
df = df.dropna(subset=['lag_1h', 'lag_24h', 'lag_168h'])
print(f'dropped {before - len(df):,} rows missing lags, {len(df):,} remain')

print('\ntarget distribution:')
print(df['avg_available'].describe().round(2))

df.head(3)

## 2. train / val / test split

time-based split. never mix future data into training. using a random split on time series data would leak future information and make results look better than they really are.

- test: last 6 weeks. held out completely, used only for final evaluation
- val: 2 weeks before test. used only for early stopping, not for parameter choices
- train: everything before val

early stopping trains up to 1000 trees but stops when validation loss stops improving. this prevents overfitting without manually tuning the number of trees.

In [ ]:
TEST_WEEKS = 6
VAL_WEEKS  = 2

test_start = df['hour'].max() - pd.Timedelta(weeks=TEST_WEEKS)
val_start  = test_start - pd.Timedelta(weeks=VAL_WEEKS)

train = df[df['hour'] < val_start].copy()
val   = df[(df['hour'] >= val_start) & (df['hour'] < test_start)].copy()
test  = df[df['hour'] >= test_start].copy()

print(f'train: {len(train):>8,} rows  ({train["hour"].min().date()} — {train["hour"].max().date()})')
print(f'val:   {len(val):>8,} rows  ({val["hour"].min().date()} — {val["hour"].max().date()})')
print(f'test:  {len(test):>8,} rows  ({test["hour"].min().date()} — {test["hour"].max().date()})')

## 3. lightgbm model

lightgbm is a gradient boosting algorithm. it builds many small decision trees one by one, where each tree corrects the errors of the previous one. it is fast, handles missing values natively, and works well on tabular data with mixed feature types.

we use it as the primary model because it handles the combination of categorical features (district, weathercode), continuous features (lat, lng, temperature), and cyclical time features (hour_sin, hour_cos) without needing separate preprocessing for each type.

xgboost and random forest are included in the comparison to confirm that the lightgbm choice is justified.

In [ ]:
GEO_COLS = [c for c in df.columns if c.startswith(('dist_', 'n_metro', 'n_tram', 'n_cafe', 'n_park', 'n_office', 'elevation'))]

FEATURE_COLS = [
    'lat', 'lng', 'bike_racks', 'district',
    *GEO_COLS,
    # season removed — near-zero importance (barely any cold-month data, so it's almost constant)
    'hour_of_day', 'dow', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    # is_raining and is_snowing removed — redundant with precipitation and snowfall
    'temperature', 'precipitation', 'windspeed', 'weathercode', 'snowfall',
    'lag_1h', 'lag_24h', 'lag_168h',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
TARGET = 'avg_available'

print(f'features: {len(FEATURE_COLS)} columns')
if GEO_COLS:
    print(f'  geo: {GEO_COLS}')

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET]

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    force_col_wise=True,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100),
    ],
)

print(f'\nbest iteration: {model.best_iteration_}')
print()
for split_name, X, y in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    preds = model.predict(X)
    mae  = mean_absolute_error(y, preds)
    rmse = mean_squared_error(y, preds) ** 0.5
    r2   = r2_score(y, preds)
    print(f'{split_name:6s}  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.3f}')

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
importances.plot(kind='barh', ax=ax, color='#3498db')
ax.set_title('feature importance (lgbm split count)')
ax.set_xlabel('importance')
plt.tight_layout()
plt.show()

split count importance shows how many times each feature was used to make a split across all trees. it reflects which features the model found useful for grouping data, but it can be biased — features with more unique values tend to get more splits regardless of actual impact. the permutation importance section below gives a more reliable view.

## 4. model comparison

comparing lightgbm with different feature sets shows what each group of features actually adds. we also compare against xgboost, random forest, and a naive baseline to confirm lightgbm is the right choice.

the naive baseline predicts last week's same hour (lag_168h). if the model cannot beat this, there is no point in using ml at all.

all tree models use geo and weather features. the four lightgbm rows isolate the contribution of each feature group.

In [ ]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

WEATHER_COLS = ['temperature', 'precipitation', 'windspeed', 'weathercode', 'snowfall']
GEO_COLS_SET = set(GEO_COLS)

# four feature sets to show what each layer adds
FEAT_CORE    = [c for c in FEATURE_COLS if c not in WEATHER_COLS and c not in GEO_COLS_SET]
FEAT_GEO     = [c for c in FEATURE_COLS if c not in WEATHER_COLS]
FEAT_WEATHER = [c for c in FEATURE_COLS if c not in GEO_COLS_SET]
FEAT_FULL    = FEATURE_COLS  # geo + weather

results = []

# naive baseline: predict same-hour-last-week
baseline_pred = test['lag_168h'].fillna(test['lag_24h']).fillna(y_test.mean())
results.append({'model': 'naive (lag 168h)', 'features': 'none',
                'test_mae': mean_absolute_error(y_test, baseline_pred),
                'test_r2':  r2_score(y_test, baseline_pred)})

def quick_lgb(feats, n_trees):
    m = lgb.LGBMRegressor(n_estimators=n_trees, learning_rate=0.05, num_leaves=64,
                          min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                          force_col_wise=True, random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_train[feats], y_train)
    return m.predict(X_test[feats])

n = model.best_iteration_

for label, desc, feats in [
    ('lightgbm', 'time + lag',             FEAT_CORE),
    ('lightgbm', '+ geo',                  FEAT_GEO),
    ('lightgbm', '+ weather',              FEAT_WEATHER),
    ('lightgbm', '+ geo + weather (full)', FEAT_FULL),
]:
    pred = model.predict(X_test) if feats is FEAT_FULL else quick_lgb(feats, n)
    results.append({'model': label, 'features': desc,
                    'test_mae': mean_absolute_error(y_test, pred),
                    'test_r2':  r2_score(y_test, pred)})

# xgboost + rf on full features for comparison
xgb = XGBRegressor(n_estimators=n, learning_rate=0.05, max_depth=6,
                   subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
pred_xgb = xgb.predict(X_test)
results.append({'model': 'xgboost', 'features': '+ geo + weather',
                'test_mae': mean_absolute_error(y_test, pred_xgb),
                'test_r2':  r2_score(y_test, pred_xgb)})

rf = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=10,
                           n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
results.append({'model': 'random forest', 'features': '+ geo + weather',
                'test_mae': mean_absolute_error(y_test, pred_rf),
                'test_r2':  r2_score(y_test, pred_rf)})

comparison = pd.DataFrame(results)
comparison[['test_mae', 'test_r2']] = comparison[['test_mae', 'test_r2']].round(3)
display(comparison)

the simplest lightgbm model (time + lag features only) performs best. this is expected for 1h-ahead prediction: lag_1h already tells the model the current station state, which explains most of the variance. adding geo and weather features increases the feature space without adding much new predictive signal for a 1-hour horizon. weather matters more for longer forecasts (next day, next week) where the lag features are not available.

the naive baseline gets a negative r2, meaning it is worse than simply predicting the global average for every station at every hour. this happens when last week's value is a poor proxy for the current week — for example, if an unusual event caused high or low demand that week. the ml models beat it by a large margin.

mae of ~0.12 bikes means the model is on average off by 0.12 bikes. since the target ranges from 0 to ~15, this is a very accurate prediction for a 1h horizon. mae is in the same unit as the target (bikes), which makes it easy to interpret operationally.

## 5. evaluation

three graphs below show different aspects of model quality:

- **actual vs predicted**: each dot is one test observation. a perfect model would put all dots on the red diagonal line. the closer the scatter is to the diagonal, the better.
- **prediction bias by hour**: mean(predicted - actual) per hour of day. positive means the model overpredicts at that hour, negative means it underpredicts. some bias at peak hours is expected — demand spikes are harder to capture precisely.
- **mean residual heatmap**: same idea split by day of week and hour. mostly white means near-zero residuals — the model has no strong systematic blind spots.

In [ ]:
test = test.copy()
test['pred']     = model.predict(X_test)
test['residual'] = test['pred'] - test[TARGET]

# actual vs predicted
sample = test.sample(min(5000, len(test)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(sample[TARGET], sample['pred'], alpha=0.15, s=8, color='#3498db')
lim = max(sample[TARGET].max(), sample['pred'].max()) * 1.05
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1)
axes[0].set_xlabel('actual avg_available')
axes[0].set_ylabel('predicted')
axes[0].set_title('actual vs predicted (test set sample)')

# mean bias by hour of day
res_hour = test.groupby('hour_of_day')['residual'].mean().reset_index()
axes[1].bar(res_hour['hour_of_day'], res_hour['residual'], color='#e74c3c', alpha=0.8)
axes[1].axhline(0, color='k', linewidth=0.8)
axes[1].set_xlabel('hour of day')
axes[1].set_ylabel('mean residual (pred - actual)')
axes[1].set_title('prediction bias by hour')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

In [ ]:
# bootstrap 95% confidence interval for test MAE
# resamples the test set 1000 times to estimate how stable the metric is
rng = np.random.default_rng(42)
boot_mae = [
    mean_absolute_error(y_test.values[idx], test['pred'].values[idx])
    for idx in (rng.choice(len(y_test), size=len(y_test)) for _ in range(1000))
]
ci_lo, ci_hi = np.percentile(boot_mae, [2.5, 97.5])

mae_test  = mean_absolute_error(y_test, test['pred'])
rmse_test = mean_squared_error(y_test, test['pred']) ** 0.5
r2_test   = r2_score(y_test, test['pred'])

print(f'test MAE:   {mae_test:.4f}   95% CI: ({ci_lo:.4f}, {ci_hi:.4f})')
print(f'test RMSE:  {rmse_test:.4f}')
print(f'test R²:    {r2_test:.4f}')
print()
print('MAE = average error in bikes. 0.12 means on average the prediction is off by 0.12 bikes.')
print('RMSE penalises large errors more — if RMSE >> MAE, some stations have occasional big spikes.')

In [ ]:
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

res_hm = test.groupby(['dow', 'hour_of_day'])['residual'].mean().reset_index()
res_pivot = res_hm.pivot(index='dow', columns='hour_of_day', values='residual')
res_pivot.index = [dow_labels[i] for i in res_pivot.index]

# use 90th percentile of absolute residuals as the cap so the colorbar
# shows real variation without being dominated by a handful of outlier stations
cap = float(np.percentile(np.abs(res_pivot.values[~np.isnan(res_pivot.values)]), 90))
cap = max(cap, 0.1)  # never less than 0.1 so color still shows

actual_max = float(np.abs(res_pivot.values).max())
print(f'residual range: {res_pivot.values.min():.3f} to {res_pivot.values.max():.3f}')
print(f'colorbar cap: ±{cap:.2f}  (90th pct of abs residuals; actual max {actual_max:.1f})')

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(res_pivot, ax=ax, cmap='RdBu_r', center=0, vmin=-cap, vmax=cap,
            cbar_kws={'label': 'mean residual (pred − actual)'})
ax.set_title(f'mean prediction residual by day and hour (test set) | cap ±{cap:.2f}')
ax.set_xlabel('hour of day')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

# if positive (red) → model overpredicts at those hours (predicts more bikes than reality)
# if negative (blue) → model underpredicts (reality had fewer bikes than predicted)
print('\nmost overpredicted hour:', res_hm.loc[res_hm["residual"].idxmax(), ["dow","hour_of_day","residual"]].to_dict())
print('most underpredicted hour:', res_hm.loc[res_hm["residual"].idxmin(), ["dow","hour_of_day","residual"]].to_dict())

In [ ]:
per_station = (
    test.groupby('station_uid')
    .apply(lambda g: pd.Series({
        'mae':  mean_absolute_error(g[TARGET], g['pred']),
        'rmse': mean_squared_error(g[TARGET], g['pred']) ** 0.5,
        'n':    len(g),
    }))
    .reset_index()
    .sort_values('mae', ascending=False)
)

print(f'median station MAE:  {per_station["mae"].median():.3f}')
print(f'mean station MAE:    {per_station["mae"].mean():.3f}')
print(f'worst 10 stations:')
display(per_station.head(10))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(per_station['mae'], bins=40, color='#3498db', edgecolor='none')
ax.axvline(per_station['mae'].median(), color='r', linestyle='--',
           label=f'median {per_station["mae"].median():.2f}')
ax.set_xlabel('MAE per station')
ax.set_title('per-station MAE distribution (test set)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. per-station analysis

the overall mae hides a lot of variation. most stations are predicted well but a small number of stations are much harder to predict — typically high-traffic hubs with irregular rebalancing activity.

the histogram shows how mae is distributed across all stations. the table shows the 10 worst-predicted stations. n = number of test observations for that station (stations with few observations may have less reliable mae estimates).

rmse penalises large errors more than mae (because errors are squared before averaging). if rmse is much higher than mae for a station, it means occasional large spikes rather than consistent small errors.

## 5. permutation importance

permutation importance asks: "if i randomly shuffle the values of one feature and re-run predictions, how much does MAE increase?" a big increase means the model relied heavily on that feature. this is more reliable than split-count importance because it measures actual impact on the metric, not just how often the feature was used to build a tree node.

features with higher increase in mae are more important. the error bars show variance across the 5 repetitions — a feature with small error bars has a stable effect. features with near-zero impact can be safely removed without affecting model quality.

In [ ]:
from sklearn.inspection import permutation_importance as sk_pi

# use 5000 test rows for speed — full test set would take many minutes
pi_idx = np.random.default_rng(42).choice(len(X_test), size=min(5000, len(X_test)), replace=False)
X_pi   = X_test.iloc[pi_idx]
y_pi   = y_test.values[pi_idx]

pi_result = sk_pi(
    model, X_pi, y_pi,
    scoring='neg_mean_absolute_error',
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

pi_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': pi_result.importances_mean,
    'std':        pi_result.importances_std,
}).sort_values('importance', ascending=True)

# only show features with non-trivial importance
top_pi = pi_df[pi_df['importance'] > 0.001]

fig, ax = plt.subplots(figsize=(9, max(4, len(top_pi) * 0.35)))
ax.barh(top_pi['feature'], top_pi['importance'], xerr=top_pi['std'],
        color='#3498db', capsize=3, error_kw={'linewidth': 0.8})
ax.set_xlabel('increase in MAE when feature is shuffled')
ax.set_title('permutation importance (test sample, n=5000)')
plt.tight_layout()
plt.show()

## 7. save model

save the trained model and the list of feature columns. the api loads both at startup — the feature list tells it which columns to build and in what order before passing data to the model.

In [ ]:
out_path = pathlib.Path('../data/model.lgb')
out_path.parent.mkdir(exist_ok=True)
model.booster_.save_model(str(out_path))
print(f'saved to {out_path}  ({out_path.stat().st_size / 1e3:.0f} KB)')

# also save the feature column list so the api can reconstruct input
import json
(out_path.parent / 'feature_cols.json').write_text(json.dumps(FEATURE_COLS))
print(f'feature list saved to {out_path.parent / "feature_cols.json"}')